Test

In [1]:
import pandas as pd

In [5]:
df_1 = pd.read_excel('/Users/hendrikalbrecht/Dokumente/Z_Übungen/Mavenanalytics/Manufacturing Downtime/Manufacturing_Line_Productivity.xlsx', sheet_name='Line productivity')

In [6]:
df_2 = pd.read_excel('/Users/hendrikalbrecht/Dokumente/Z_Übungen/Mavenanalytics/Manufacturing Downtime/Manufacturing_Line_Productivity.xlsx', sheet_name='Products')
df_3 = pd.read_excel('/Users/hendrikalbrecht/Dokumente/Z_Übungen/Mavenanalytics/Manufacturing Downtime/Manufacturing_Line_Productivity.xlsx', sheet_name='Downtime factors')
df_4 = pd.read_excel('/Users/hendrikalbrecht/Dokumente/Z_Übungen/Mavenanalytics/Manufacturing Downtime/Manufacturing_Line_Productivity.xlsx', sheet_name='Line downtime')

In [3]:
df_1.head()

,Date,Product,Batch,Operator,Start Time,End Time
0,2024-08-29,OR-600,422111,Mac,11:50:00,14:05:00
1,2024-08-29,LE-600,422112,Mac,14:05:00,15:45:00
2,2024-08-29,LE-600,422113,Mac,15:45:00,17:35:00
3,2024-08-29,LE-600,422114,Mac,17:35:00,19:15:00
4,2024-08-29,LE-600,422115,Charlie,19:15:00,20:39:00


In [7]:
df_2.head()

,Product,Flavor,Size,Min batch time
0,OR-600,Orange,600 ml,60
1,LE-600,Lemon lime,600 ml,60
2,CO-600,Cola,600 ml,60
3,DC-600,Diet Cola,600 ml,60
4,RB-600,Root Beer,600 ml,60


In [8]:
df_3.head()

,Factor,Description,Operator Error
0,1,Emergency stop,No
1,2,Batch change,Yes
2,3,Labeling error,No
3,4,Inventory shortage,No
4,5,Product spill,Yes


In [9]:
df_4.head()

,Unnamed: 0,Downtime factor,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12
0,Batch,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0
1,422111,NaN,60.0,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,NaN,NaN
2,422112,NaN,20.0,NaN,NaN,NaN,NaN,NaN,20.0,NaN,NaN,NaN,NaN
3,422113,NaN,50.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,422114,NaN,NaN,NaN,25.0,NaN,15.0,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# 1. Erste Zeile aus df_4 als neue Spaltennamen setzen
df_4_clean = df_4.copy()

df_4_clean.columns = df_4_clean.iloc[0]
df_4_clean = df_4_clean.iloc[1:].reset_index(drop=True)

# 2. Faktor-Mapping aus df_3 erstellen
factor_map = dict(zip(df_3["Factor"], df_3["Description"]))

# 3. Spaltennamen vorbereiten und umbenennen
new_columns = []

for col in df_4_clean.columns:
    # Batch-Spalte behalten
    if col == "Batch":
        new_columns.append("Batch")
    else:
        # aus 1.0 -> 1 machen
        factor_number = int(float(col))
        new_columns.append(factor_map[factor_number])

df_4_clean.columns = new_columns

# 4. Werte numerisch machen, außer Batch
for col in df_4_clean.columns:
    if col != "Batch":
        df_4_clean[col] = pd.to_numeric(df_4_clean[col], errors="coerce")

df_4_clean.head()

,Batch,Emergency stop,Batch change,Labeling error,Inventory shortage,Product spill,Machine adjustment,Machine failure,Batch coding error,Conveyor belt jam,Calibration error,Label switch,Other
0,422111,NaN,60.0,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,NaN,NaN
1,422112,NaN,20.0,NaN,NaN,NaN,NaN,NaN,20.0,NaN,NaN,NaN,NaN
2,422113,NaN,50.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,422114,NaN,NaN,NaN,25.0,NaN,15.0,NaN,NaN,NaN,NaN,NaN,NaN
4,422115,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.0,NaN,NaN


In [12]:
df_4_clean.isna().sum()

Batch                  0
Emergency stop        38
Batch change          33
Labeling error        36
Inventory shortage    29
Product spill         35
Machine adjustment    26
Machine failure       27
Batch coding error    32
Conveyor belt jam     37
Calibration error     35
Label switch          35
Other                 32
dtype: int64

In [13]:
df_4_clean["Total_Downtime"] = df_4_clean.drop(columns="Batch").sum(axis=1)

df_4_clean.head()

,Batch,Emergency stop,Batch change,Labeling error,Inventory shortage,Product spill,Machine adjustment,Machine failure,Batch coding error,Conveyor belt jam,Calibration error,Label switch,Other,Total_Downtime
0,422111,NaN,60.0,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,NaN,NaN,75.0
1,422112,NaN,20.0,NaN,NaN,NaN,NaN,NaN,20.0,NaN,NaN,NaN,NaN,40.0
2,422113,NaN,50.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0
3,422114,NaN,NaN,NaN,25.0,NaN,15.0,NaN,NaN,NaN,NaN,NaN,NaN,40.0
4,422115,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.0,NaN,NaN,24.0


Letzte Spalte von df_4_clean zu df_1 addieren/kopieren

In [14]:
df_1['Total_Downtime'] = df_4_clean['Total_Downtime']

In [15]:
df_1.columns

Index(['Date', 'Product', 'Batch', 'Operator', 'Start Time', 'End Time',
       'Total_Downtime'],
      dtype='str')

In [16]:
df_1.head()

,Date,Product,Batch,Operator,Start Time,End Time,Total_Downtime
0,2024-08-29,OR-600,422111,Mac,11:50:00,14:05:00,75.0
1,2024-08-29,LE-600,422112,Mac,14:05:00,15:45:00,40.0
2,2024-08-29,LE-600,422113,Mac,15:45:00,17:35:00,50.0
3,2024-08-29,LE-600,422114,Mac,17:35:00,19:15:00,40.0
4,2024-08-29,LE-600,422115,Charlie,19:15:00,20:39:00,24.0


#

1. Frage Line efficiency Groupby Product
3. Frage Leading factors -> df_4_clean spalten addieren, dann absteigend sortieren (grösstes zuerst)
2. Frage operators underperforming (nach operators gruppieren (mac, charlie), avg downtime, vergleichen mit avg., dann gucken wer unter und wer über avg. liegt
4. Frage Do operators struggle with particular operator error | operator error J/N (df_3) | df_4 zwei neue spalten J oder N, füllen mit der jeweiligen downtime | zuordnen wieviel zeit operator downtime bzw. maschinelle downtime | df_1 breakdown 75 minuten total downtime (anteil maschinell, anteil human) | per operator gruppieren (mac 100 minuten human, 10 minuten maschinell)
5. Natalia macht Frage 1 + 3, Hendrik macht 2 + 4
